In [22]:
%load_ext autoreload
%load_ext kedro.ipython
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The kedro.ipython extension is already loaded. To reload it, use:
  %reload_ext kedro.ipython


In [23]:
from project.example_addition import addition

print(addition(5, 5))

10


In [ ]:
from IPython import get_ipython
from kedro.io import DataCatalog

catalog: DataCatalog = get_ipython().user_ns["catalog"]
# Строчки выше можно удалить, каталог будет доступен и функционировать, 
# но Pylance будет подчеркивать его как несуществующий, так как VS Code не видит,
# где объявлена переменная catalog. Ее создает расширение kedro.ipython.
df = catalog.load("source@raw")

[11/20/26 16:19:14] INFO     Loading data from source@raw (CSVDataset)...                      ]8;id=7967730;file:///Users/sergejsvetsov/Git/python-project-template/.venv/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=7967731;file:///Users/sergejsvetsov/Git/python-project-template/.venv/lib/python3.12/site-packages/kedro/io/data_catalog.py#1151\1151]8;;\

In [25]:
from project.modeling.train import train_model, create_model, create_scaler
from project.modeling.split_data import split_data
from project.modeling.validation import cross_validate_model

X_train, X_test, y_train, y_test = split_data(df, target_col="MEDV")

model = create_model(scaler="standard", alpha=0.1, l1_ratio=0.5, max_iter=10000)


cv_rmse = cross_validate_model(
    model=model,
    X_train=X_train,
    y_train=y_train,
)
print(f"CV MAE: {cv_rmse:.3f}")

CV MAE: 3.371


In [26]:
from project.modeling.tuning import tune_model

search_space = {
    "alpha": {
        "low": 0.0001,
        "high": 10.0
    },
    "l1_ratio": {
        "low": 0.0,
        "high": 1.0
    }
}

study = tune_model(
    X_train=X_train,
    y_train=y_train,
    search_space=search_space,
    scaler="standard",
    n_trials=30,
    n_splits=5,
)

print("Лучшие параметры:", study.best_params)
print("Лучший CV RMSE:", study.best_value)

[I 2026-11-20 16:19:14,160] A new study created in memory with name: no-name-996639e9-1222-44e5-872c-7c23ede22364
[I 2026-11-20 16:19:14,170] Trial 0 finished with value: 3.4206354952366977 and parameters: {'alpha': 0.0074593432857265485, 'l1_ratio': 0.9507143064099162}. Best is trial 0 with value: 3.4206354952366977.
[I 2026-11-20 16:19:14,178] Trial 1 finished with value: 3.5316769037249416 and parameters: {'alpha': 0.45705630998014507, 'l1_ratio': 0.5986584841970366}. Best is trial 0 with value: 3.4206354952366977.
[I 2026-11-20 16:19:14,187] Trial 2 finished with value: 3.4270153550941993 and parameters: {'alpha': 0.0006026889128682511, 'l1_ratio': 0.15599452033620265}. Best is trial 0 with value: 3.4206354952366977.
[I 2026-11-20 16:19:14,195] Trial 3 finished with value: 3.42815435228052 and parameters: {'alpha': 0.00019517224641449495, 'l1_ratio': 0.8661761457749352}. Best is trial 0 with value: 3.4206354952366977.
[I 2026-11-20 16:19:14,203] Trial 4 finished with value: 3.37447

Лучшие параметры: {'alpha': 0.08775646984512295, 'l1_ratio': 0.1791123637589616}
Лучший CV RMSE: 3.3655101758014276


In [27]:
best_model = train_model(
    X_train=X_train,
    y_train=y_train,
    scaler="standard",
    max_iter=10_000,
    **study.best_params,
)

In [28]:
from project.modeling.evaluate import evaluate_model

test_metrics = evaluate_model(
    model=best_model,
    X_test=X_test,
    y_test=y_test,
)

print(test_metrics)

{'mae': {'value': 3.1660739477721167, 'step': 0}, 'rmse': {'value': 4.992062070695953, 'step': 0}, 'r2': {'value': 0.6601745792495206, 'step': 0}}
